# Pronóstico mensual de tarifas spot de camión refrigerado por lane
### Entregable 3: Obtención, análisis exploratorio y preparación de datos

**Maestría en Inteligencia Artificial Aplicada**
Inteligencia Artificial y Aprendizaje Automático (TC4058.2)
**Equipo 1**

| Integrante | Matrícula |
|---|---|
| Jaime Martínez Vogel | A01283238 |
| Héctor de Luna Aguilar | A01610988 |

### Descripción del caso
Nuestro caso es el de un distribuidor de frutas y verduras frescas que mueve producto en camiones refrigerados desde zonas agrícolas de Estados Unidos y México hacia las principales ciudades de consumo. Cuando la carga se mueve en el mercado spot, la tarifa cambia cada semana, lo cual complica armar el presupuesto de flete y decidir en qué momento conviene más amarrar un contrato con un transportista.

Por lo tanto, la pregunta que queremos contestar es **¿cuál será la tarifa spot (USD por carga) del mes siguiente para cada lane?**, entendiendo una lane como la combinación de un origen y un destino. Se trata de un problema de regresión sobre series de tiempo, ya que tenemos muchas lanes y cada una tiene su propia historia mensual.

### Estructura del notebook
1. Bibliotecas
2. Carga y procedencia de los datos
3. Exploración inicial y diccionario de variables
4. Análisis exploratorio (EDA)
5. Diagnóstico de calidad
6. Limpieza y preparación
7. Variables para el modelado y fuga de información
8. Conclusiones
9. Referencias

## 1. Bibliotecas

Para correr el notebook se necesita Python 3.11 o superior. Las bibliotecas, con las versiones exactas con las que lo probamos, están en el archivo "requirements.txt" del repositorio y se instalan con:

```
pip install -r requirements.txt
```

La biblioteca "xlrd" solo se usa para leer el archivo de Excel del diésel que publica la EIA. Si el notebook se abre en Google Colab, la siguiente celda la instala automáticamente.

In [ ]:
# Solo aplica en Google Colab, que no trae instalada la biblioteca xlrd
import sys
if "google.colab" in sys.modules:
    get_ipython().run_line_magic("pip", "install -q xlrd")

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4.5)

## 2. Carga y procedencia de los datos

### 2.1 Fuentes utilizadas

| | Fuente principal | Fuente complementaria |
|---|---|---|
| Nombre | Refrigerated Truck Rates and Availability | Weekly U.S. No 2 Diesel Retail Prices |
| Publicador | USDA, Agricultural Marketing Service (portal AgTransport) | U.S. Energy Information Administration (EIA) |
| Acceso | API pública, sin registro: https://agtransport.usda.gov/resource/acar-e3r8.json | Archivo Excel público: https://www.eia.gov/dnav/pet/hist_xls/EMD_EPD2D_PTE_NUS_DPGw.xls |
| Licencia | Datos abiertos del gobierno de EE. UU., se pueden usar citando la fuente | Dominio público, se pide citar a la EIA |
| Descripción | Tarifas semanales de camión refrigerado (mínima, máxima y punto medio en USD por carga), distancia, tarifa por milla y disponibilidad de camiones por origen, destino y producto | Precio promedio semanal del diésel en EE. UU. (USD por galón) |
| Periodo | Semanal, del 4 de enero de 2000 al 27 de octubre de 2026 | Semanal, de marzo de 1994 a octubre de 2026 |
| Tamaño | 207,024 registros y 15 columnas (al 1 de noviembre de 2026) | Alrededor de 1,700 semanas y 2 columnas |
| Variable objetivo | "midpoint", que convertimos en la mediana mensual por lane | No aplica, se usa como predictor |

Ninguna de las dos fuentes contiene datos personales ni de empresas, ya que son promedios de mercado.

Decidimos cargar los datos de USDA directamente desde su API en lugar de bajar un archivo a mano, porque así cualquier persona puede reproducir el notebook y además podemos consultar al servidor cuántos registros tiene, para confirmar que la descarga quedó completa. Las copias descargadas se guardan en la carpeta "data/raw" y no se modifican, por lo que funcionan como nuestro dataset original. El archivo de USDA se guarda comprimido (".csv.gz") para que pese alrededor de una décima parte y se pueda subir al repositorio; además, como el notebook siempre usa primero la copia local, puede correr completo aunque no haya internet o la API no esté disponible.

In [ ]:
# Carpetas relativas al notebook (no usamos rutas personales)
CARPETA_RAW = Path("data/raw")
CARPETA_PROC = Path("data/processed")
CARPETA_RAW.mkdir(parents=True, exist_ok=True)
CARPETA_PROC.mkdir(parents=True, exist_ok=True)

URL_USDA = "https://agtransport.usda.gov/resource/acar-e3r8.json"
URL_USDA_META = "https://agtransport.usda.gov/api/views/acar-e3r8.json"
URL_DIESEL = "https://www.eia.gov/dnav/pet/hist_xls/EMD_EPD2D_PTE_NUS_DPGw.xls"

ARCHIVO_USDA = CARPETA_RAW / "usda_refrigerated_truck_rates_raw.csv.gz"
ARCHIVO_META = CARPETA_RAW / "usda_metadata.json"
ARCHIVO_DIESEL = CARPETA_RAW / "eia_diesel_semanal_raw.xls"

# Algunos servidores no contestan si la petición no parece venir de un navegador
HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36"}

### 2.2 Cuántos registros tiene el servidor

Antes de descargar le preguntamos a la API cuántos registros tiene. La API acepta consultas en SoQL, que funciona parecido a SQL, así que el conteo lo hace el servidor y nosotros solo recibimos el resultado.

In [ ]:
try:
    respuesta = requests.get(URL_USDA, params={"$select": "count(*) AS registros"}, headers=HEADERS, timeout=30)
    total_servidor = int(respuesta.json()[0]["registros"])
    print(f"Registros en el servidor de USDA: {total_servidor:,}")
except requests.RequestException:
    total_servidor = None
    print("No hay conexión con la API, se trabajará con la copia local")

### 2.3 Descarga de los datos de USDA

La API entrega los datos por bloques, por lo que los bajamos en páginas de 50,000 registros, moviendo el punto de inicio ("$offset") en cada vuelta y ordenando por el identificador interno (":id") para que no se repitan ni se pierdan filas entre páginas. Si la copia local ya existe, no se vuelve a descargar.

In [ ]:
TAM_PAGINA = 50000

if ARCHIVO_USDA.exists():
    print("Usando la copia local que ya existe")
elif total_servidor is None:
    raise RuntimeError("No hay copia local ni conexión con la API: revisa tu internet y vuelve a correr la celda anterior")
else:
    paginas = []
    offset = 0
    while offset < total_servidor:
        params = {"$limit": TAM_PAGINA, "$offset": offset, "$order": ":id"}
        respuesta = requests.get(URL_USDA, params=params, headers=HEADERS, timeout=300)
        paginas.append(pd.DataFrame(respuesta.json()))
        offset += TAM_PAGINA
        print(f"Descargados {min(offset, total_servidor):,} de {total_servidor:,}")
        time.sleep(0.5)
    pd.concat(paginas).to_csv(ARCHIVO_USDA, index=False)

# Se lee todo como texto para revisar los tipos en la sección 5
df_raw = pd.read_csv(ARCHIVO_USDA, dtype=str)
print(f"Registros en la copia local: {len(df_raw):,}")
if total_servidor is not None:
    print("¿Coincide con el servidor?", len(df_raw) == total_servidor)

Si el resultado de la comparación es False, lo más probable es que USDA haya agregado la semana más reciente después de que hicimos la copia local, ya que el dataset se actualiza cada semana. En ese caso basta con borrar el archivo de "data/raw" y volver a correr la celda.

### 2.4 Descarga del precio del diésel

In [ ]:
if not ARCHIVO_DIESEL.exists():
    respuesta = requests.get(URL_DIESEL, headers=HEADERS, timeout=120)
    ARCHIVO_DIESEL.write_bytes(respuesta.content)

# La hoja "Data 1" tiene dos renglones de título antes de los encabezados
diesel_raw = pd.read_excel(ARCHIVO_DIESEL, sheet_name="Data 1", skiprows=2)

### 2.5 Vista inicial

In [ ]:
df_raw.shape

In [ ]:
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
print(diesel_raw.shape)
diesel_raw.head()

In [ ]:
diesel_raw.info()

Cada renglón del dataset de USDA es la tarifa de una semana para una combinación de origen, destino y producto. Lo primero que notamos es que la API entrega las 15 columnas como texto, incluso las fechas y los montos, por lo que antes de analizar hay que convertir cada columna a su tipo correcto. Por otro lado, el archivo del diésel solo tiene dos columnas, la fecha de la semana y el precio por galón.

## 3. Exploración inicial y diccionario de variables

### 3.1 Descripciones oficiales desde la API

La API también publica los metadatos del dataset, así que tomamos de ahí la descripción oficial de cada columna para armar el diccionario. Igual que con los datos, la primera vez se descargan y se guardan en "data/raw", y las siguientes veces se leen del disco.

In [ ]:
if not ARCHIVO_META.exists():
    respuesta = requests.get(URL_USDA_META, headers=HEADERS, timeout=60)
    ARCHIVO_META.write_text(respuesta.text, encoding="utf-8")
meta = json.loads(ARCHIVO_META.read_text(encoding="utf-8"))

descripciones = pd.DataFrame(
    [(c["fieldName"], c["dataTypeName"], c.get("description", "")) for c in meta["columns"]
     if not c["fieldName"].startswith(":")],
    columns=["columna", "tipo_en_usda", "descripcion_oficial"],
)
descripciones

### 3.2 Conversión de tipos

Trabajamos sobre una copia ("df") para no modificar el original. Usamos errors="coerce" para que, si algún valor no se puede convertir a número, quede como faltante en vez de detener el código.

In [ ]:
df = df_raw.copy()
df["date"] = pd.to_datetime(df["date"])

columnas_numericas = ["week", "month", "quarter", "year", "distance",
                      "weeklow", "weekhigh", "midpoint", "rpm", "availability"]
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Llave de cada serie: la lane (origen y destino)
df["lane"] = df["origin"] + " → " + df["destination"]

df.dtypes

In [ ]:
diesel = diesel_raw.iloc[:, :2].copy()
diesel.columns = ["fecha", "diesel"]
diesel["fecha"] = pd.to_datetime(diesel["fecha"], errors="coerce")
diesel["diesel"] = pd.to_numeric(diesel["diesel"], errors="coerce")
diesel = diesel.dropna()
diesel.head()

In [ ]:
df[["origin", "destination", "region", "commodity", "lane"]].nunique()

### 3.3 Diccionario de datos

| Columna | Tipo | Significado | Papel en el modelo |
|---|---|---|---|
| "date" | Fecha | Fecha de cierre de la semana reportada | Llave temporal, se agrupa por mes |
| "week", "month", "quarter", "year" | Entero | Semana, mes, trimestre y año calculados por USDA a partir de "date" | Solo usamos año y mes; "week" y "quarter" son redundantes |
| "region" | Texto | Región a la que USDA asigna cada origen | No se usa, ya que repite la información de "origin" y además genera duplicados (sección 5) |
| "origin" | Texto | Distrito de origen de la carga | Parte de la lane, entrada del modelo |
| "destination" | Texto | Una de 10 ciudades destino | Parte de la lane, entrada del modelo |
| "distance" | Numérica | Distancia estimada entre origen y destino, en millas | Entrada, ya que es un atributo fijo de la lane |
| "commodity" | Texto | Producto o mezcla de productos a la que aplica la tarifa | Auxiliar, ya que agregamos todos los productos de una lane |
| "weeklow", "weekhigh" | Numérica | Tarifa mínima y máxima de la semana en USD por carga | No se usan, porque se publican al mismo tiempo que el target |
| "midpoint" | Numérica | Tarifa representativa de la semana en USD por carga | **Target**, como mediana mensual por lane |
| "rpm" | Numérica | Tarifa por milla, es decir "midpoint" entre "distance" | No se usa como entrada porque contiene al target; sí nos sirve para comparar lanes en el EDA |
| "availability" | Ordinal (1 a 5) | Disponibilidad de camiones: 1 es excedente, 3 es adecuada y 5 es escasez | Entrada, pero con un mes de rezago |
| "diesel" (EIA) | Numérica | Precio promedio del diésel en USD por galón | Entrada, con un mes de rezago |

En resumen, el target es la tarifa mensual de cada lane, las entradas potenciales son la distancia, la lane, el calendario, los valores pasados de la tarifa, la disponibilidad y el diésel del mes anterior, y las variables que no conviene usar son "weeklow", "weekhigh" y "rpm", ya que revelan la tarifa del mismo periodo que queremos predecir.

## 4. Análisis exploratorio (EDA)

### 4.1 Estadísticas descriptivas

In [ ]:
df[["distance", "weeklow", "weekhigh", "midpoint", "rpm", "availability"]].describe().round(2)

In [ ]:
df["destination"].value_counts()

El dataset cubre 10 destinos, donde Nueva York, Chicago y Atlanta son los que tienen más registros y Seattle el que menos. La tarifa "midpoint" va de 215 a 14,700 USD por carga y la tarifa por milla de 0.13 a 41.67 USD, lo cual es un rango muy amplio que se explica principalmente por la distancia de cada lane y por los años de precios altos, como se ve en las siguientes gráficas.

### 4.2 Gráfica 1: distribución de la tarifa

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(df["midpoint"], bins=60, ax=ax[0])
ax[0].axvline(df["midpoint"].median(), color="red", linestyle="--", label="mediana")
ax[0].set_title("Tarifa por carga (midpoint, USD)")
ax[0].legend()

# La tarifa por milla se corta en el percentil 99 para que la gráfica se pueda leer
limite = df["rpm"].quantile(0.99)
sns.histplot(df.loc[df["rpm"] <= limite, "rpm"], bins=60, ax=ax[1], color="darkorange")
ax[1].set_title("Tarifa por milla (rpm, hasta el percentil 99)")

plt.tight_layout()
plt.show()

Las dos distribuciones tienen asimetría positiva, es decir, la mayoría de los registros se concentra en valores medios pero hay una cola larga hacia la derecha. En el caso de la tarifa por carga, la cola corresponde a lanes largas (costa oeste a costa este) y a los años de precios altos. En el caso de la tarifa por milla, los valores extremos vienen de lanes muy cortas; por ejemplo, South New Jersey a Philadelphia mide 30 millas y cobra 1,250 USD por carga, lo que da 41.67 USD por milla, pero esto no es un error sino el efecto del cargo mínimo de un viaje. Por esta razón decidimos usar la mediana al agregar por mes y evaluar el modelo con el MAE, que es menos sensible a estos valores que el RMSE.

### 4.3 Gráfica 2: comportamiento en el tiempo

Para comparar entre años usamos la tarifa por milla y no la tarifa por carga, ya que la tarifa por carga depende de qué tan largas son las lanes que se reportan cada año.

In [ ]:
rpm_nacional = df.groupby(df["date"].dt.to_period("M"))["rpm"].median()
rpm_nacional.index = rpm_nacional.index.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(rpm_nacional.index, rpm_nacional.values)
ax.axvspan(pd.Timestamp("2020-09-01"), pd.Timestamp("2022-06-30"), color="orange", alpha=0.2, label="Choque pospandemia")
ax.set_title("Mediana mensual de la tarifa por milla (USD por milla)")
ax.legend()
plt.show()

df.groupby("year")["rpm"].mean().round(2).to_frame("rpm_promedio").T

La serie tiene una tendencia creciente con varios cambios de nivel. La tarifa por milla promedio pasó de 1.63 USD en 2000 a 2.84 en 2008, año en el que el diésel llegó a su máximo, y después bajó a 2.32 en 2009 con la recesión. El cambio más fuerte fue el choque pospandemia, donde la tarifa subió a 3.89 en 2021 y a 4.27 en 2022, con su punto más alto en enero de 2022 (4.96). Entre 2023 y 2025 el mercado se estabilizó alrededor de 3.4 a 3.5, pero en 2026 vuelve a subir y llega a 4.00 en promedio de enero a octubre (+17% contra 2025).

Esto hace que no podamos validar el modelo mezclando meses al azar, sino que hay que entrenar con el pasado y evaluar con el periodo más reciente. Adicionalmente, los modelos basados en árboles podrían tener problemas para predecir niveles más altos que los que vieron en el entrenamiento.

### 4.4 Gráfica 3: estacionalidad

Para ver el efecto del mes sin que la tendencia lo esconda, dividimos el promedio de cada mes entre el promedio de su propio año. Un valor de 1.05 significa que ese mes fue 5% más caro que el promedio del año. Dejamos fuera 2026 porque todavía está incompleto.

In [ ]:
por_mes = df.groupby(["year", "month"])["rpm"].mean().reset_index()
por_mes = por_mes[por_mes["year"] < df["year"].max()]
por_mes["indice"] = por_mes["rpm"] / por_mes.groupby("year")["rpm"].transform("mean")

sns.boxplot(data=por_mes, x="month", y="indice", color="lightblue", showfliers=False)
plt.axhline(1, color="red", linestyle="--")
plt.title("Tarifa por milla de cada mes respecto al promedio de su año")
plt.xlabel("Mes")
plt.show()

por_mes.groupby("month")["indice"].mean().round(3).to_frame("indice_promedio").T

Encontramos una estacionalidad moderada pero bastante constante: julio es en promedio 5.5% más caro que el resto de su año, mientras que febrero y marzo son alrededor de 6% más baratos, y junio y julio quedaron arriba del promedio anual en cerca del 85% de los años. Esto coincide con la temporada alta de cosecha y embarque de productos frescos en verano. Por lo tanto, vale la pena incluir el mes como variable y el rezago de 12 meses (mismo mes del año anterior), y también usar ese mismo rezago como un baseline sencillo contra el cual comparar el modelo.

### 4.5 Gráfica 4: tarifa y distancia

In [ ]:
muestra = df.dropna(subset=["distance", "midpoint"]).sample(5000, random_state=42)

sns.scatterplot(data=muestra, x="distance", y="midpoint", alpha=0.3, s=10)
plt.title("Tarifa por carga vs distancia")
plt.show()

print("Correlación distancia vs midpoint:", round(df["distance"].corr(df["midpoint"]), 3))
print("Tarifa por milla promedio en lanes de menos de 700 millas:", round(df.loc[df["distance"] < 700, "rpm"].mean(), 2))
print("Tarifa por milla promedio en lanes de 2,000 millas o más:", round(df.loc[df["distance"] >= 2000, "rpm"].mean(), 2))

La distancia explica gran parte del nivel de la tarifa por carga, ya que la relación es claramente positiva y casi lineal; la dispersión vertical que se ve para una misma distancia corresponde principalmente a los distintos años. Por otro lado, la tarifa por milla baja conforme aumenta la distancia: las lanes de menos de 700 millas cuestan en promedio 4.73 USD por milla, mientras que las de 2,000 millas o más cuestan 2.33, lo cual se debe a que los costos fijos de cada viaje (carga, descarga y cargo mínimo) se reparten entre más millas. En conclusión, la distancia o la identidad de la lane tiene que entrar al modelo.

### 4.6 Gráfica 5: disponibilidad de camiones

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

porcentaje = df["availability"].value_counts(normalize=True).sort_index() * 100
ax[0].bar(porcentaje.index.astype(int).astype(str), porcentaje.values, color="seagreen")
ax[0].set_title("Registros por nivel de availability (%)")
ax[0].set_xlabel("1 = excedente, 3 = adecuada, 5 = escasez")

sns.boxplot(data=df, x="availability", y="rpm", ax=ax[1], showfliers=False, color="lightblue")
ax[1].set_title("Tarifa por milla según disponibilidad")

plt.tight_layout()
plt.show()

df.groupby("availability")["rpm"].mean().round(2).to_frame("rpm_promedio").T

La variable está desbalanceada, ya que el 61.5% de los registros reporta una oferta adecuada (nivel 3). Cuando hay excedente de camiones (niveles 1 y 2) la tarifa por milla es más baja, con promedios de 2.17 y 2.31 USD contra 2.89 en el nivel 3, lo cual tiene sentido con la escala oficial y nos ayuda a confirmar que no está invertida, que era uno de los riesgos que identificamos en el Entregable 2. Sin embargo, entre los niveles 3, 4 y 5 casi no hay diferencia, probablemente porque el efecto de la disponibilidad se mezcla con el del año y el de la lane, por lo que no esperamos que sea de las variables más fuertes del modelo.

### 4.7 Gráfica 6: cobertura de los datos en el tiempo

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

df.groupby("year").size().plot.bar(ax=ax[0], color="gray")
ax[0].set_title("Registros por año")

df.groupby("year")["lane"].nunique().plot.bar(ax=ax[1], color="steelblue")
ax[1].set_title("Lanes con al menos un dato por año")

plt.tight_layout()
plt.show()

df.groupby("year")["distance"].mean().round(0).to_frame("distancia_promedio").T

La cantidad de información no es constante en el tiempo, ya que los registros por año bajaron de cerca de 10,000 entre 2008 y 2016 a 3,875 en 2025, mientras que la distancia promedio de lo que se reporta subió de unas 1,300 a casi 1,900 millas. Esto quiere decir que cambió la mezcla de lanes que publica USDA: algunos orígenes dejaron de reportarse (Michigan en 2022 y Eastern North Carolina en 2023, por ejemplo), otros son estacionales y solo aparecen en su temporada, y otros cambiaron de nombre. Por esta razón un promedio nacional no sirve como target, ya que podría subir o bajar solo porque cambiaron las lanes reportadas, y por eso trabajamos lane por lane.

### 4.8 Gráfica 7: tarifa y precio del diésel

In [ ]:
diesel_mensual = diesel.groupby(diesel["fecha"].dt.to_period("M"))["diesel"].mean()
diesel_mensual.index = diesel_mensual.index.to_timestamp()
comparacion = pd.concat([rpm_nacional.rename("rpm"), diesel_mensual.rename("diesel")], axis=1).dropna()

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(comparacion.index, comparacion["rpm"], label="Tarifa por milla")
ax.set_ylabel("USD por milla")
ax2 = ax.twinx()
ax2.plot(comparacion.index, comparacion["diesel"], color="darkorange", label="Diésel")
ax2.set_ylabel("USD por galón")
ax2.grid(False)
ax.set_title("Tarifa por milla y precio del diésel (promedio mensual)")
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.88))
plt.show()

print("Correlación en niveles:", round(comparacion["rpm"].corr(comparacion["diesel"]), 2))
print("Correlación en cambios mensuales:", round(comparacion["rpm"].pct_change().corr(comparacion["diesel"].pct_change()), 2))

El diésel y la tarifa se mueven juntos en los episodios más grandes, ya que ambos tienen picos a mediados de 2008 y en 2022. Sin embargo, no todo se explica con el combustible, porque en 2021 la tarifa subió mucho sin que el diésel subiera de la misma forma, lo cual apunta a la falta de capacidad de transporte de esa época. También hay que tomar en cuenta que la correlación en niveles sale alta en parte porque las dos series crecen con el tiempo, por lo que la correlación en cambios mensuales es una mejor medida de qué tanto se mueven juntas en el corto plazo. Por lo tanto, incluimos el diésel del mes anterior como entrada, sabiendo que es un promedio nacional y no uno por lane.

## 5. Diagnóstico de calidad

### 5.1 Valores faltantes

In [ ]:
faltantes = df.isna().sum()
porcentaje = (df.isna().mean() * 100).round(3)
pd.DataFrame({"faltantes": faltantes, "porcentaje": porcentaje})[faltantes > 0]

In [ ]:
print("Filas sin midpoint y sin distance al mismo tiempo:", (df["midpoint"].isna() & df["distance"].isna()).sum())

Los faltantes son pocos, ya que ninguna columna llega al 1%: hay 1,080 registros sin "availability" (0.52%) y 798 registros sin "distance" o sin "midpoint". Las filas sin "midpoint" no se pueden usar porque no tienen target, y las filas sin "distance" no permiten calcular la tarifa por milla, mientras que los faltantes de "availability" sí se pueden manejar después con imputación.

### 5.2 Registros duplicados

In [ ]:
llave = ["date", "origin", "destination", "commodity"]

print("Duplicados exactos (todas las columnas iguales):", df.duplicated().sum())
print("Duplicados por llave (misma semana, origen, destino y producto):", df.duplicated(subset=llave).sum())

df[df.duplicated(subset=llave, keep=False)].sort_values(llave).head(4)

Además de buscar filas idénticas, revisamos los duplicados por llave, es decir, casos donde la misma semana, origen, destino y producto aparece más de una vez. Al revisarlos vimos que la diferencia está en la columna "region": USDA asigna el origen del Valle Imperial y Coachella (que abarca California y Arizona) tanto a la región ARIZONA como a CALIFORNIA, con exactamente las mismas tarifas. Si los dejamos, esa lane tendría el doble de peso en los promedios y la misma información podría quedar en entrenamiento y en prueba, por lo que los eliminamos en la limpieza.

### 5.3 Tipos de dato incorrectos

Como vimos en la sección 2.5, la API entrega todo como texto. Para verificar que la conversión de la sección 3.2 no perdió información, comparamos los nulos antes y después de convertir: si aparecen nulos nuevos, significa que había valores que no eran números válidos.

In [ ]:
for col in columnas_numericas:
    nuevos_nulos = df[col].isna().sum() - df_raw[col].isna().sum()
    print(f"{col:13s} nulos generados al convertir: {nuevos_nulos}")

Como USDA guarda estas columnas como números en su base de datos, no se generan nulos nuevos al convertirlas, lo cual quiere decir que todos los textos eran números válidos y que el único problema de tipo era el formato en el que los entrega la API, que ya quedó corregido.

### 5.4 Registros inválidos

In [ ]:
fuera_de_rango = (df["midpoint"] < df["weeklow"]) | (df["midpoint"] > df["weekhigh"])
promedio_rango = (df["weeklow"] + df["weekhigh"]) / 2

print("weeklow mayor que weekhigh:", (df["weeklow"] > df["weekhigh"]).sum())
print("midpoint fuera de su rango semanal:", fuera_de_rango.sum())
print("Tarifas o distancias menores o iguales a cero:", ((df["midpoint"] <= 0) | (df["distance"] <= 0)).sum())
print("availability fuera de la escala 1 a 5:", (df["availability"].notna() & ~df["availability"].isin([1, 2, 3, 4, 5])).sum())
print("midpoint distinto al promedio de weeklow y weekhigh:", ((df["midpoint"] - promedio_rango).abs() > 0).sum())

In [ ]:
# Ejemplos de midpoint que no es el promedio del rango
ejemplos = df.loc[(df["midpoint"] - promedio_rango).abs() > 0, ["date", "lane", "weeklow", "weekhigh", "midpoint"]].head(5)
ejemplos["promedio_rango"] = (ejemplos["weeklow"] + ejemplos["weekhigh"]) / 2
ejemplos

No hay rangos invertidos, tarifas negativas ni niveles de disponibilidad fuera de la escala, pero sí hay 6 registros donde "midpoint" queda fuera de su propio rango semanal, lo cual es imposible, así que se eliminan.

El hallazgo más importante de esta revisión es que la documentación de USDA dice que "midpoint" es el promedio de "weeklow" y "weekhigh", pero en 93,543 registros (cerca del 45%) no lo es. Por ejemplo, en la lane de Yakima a Atlanta de enero de 2000 el rango es de 3,000 a 3,500 USD y el "midpoint" es 3,000. Todo indica que USDA publica la tarifa más frecuente de la semana y no el promedio aritmético, por lo que decidimos conservar el "midpoint" publicado como target, ya que es el valor que la fuente oficial considera representativo, y dejar documentada la diferencia.

### 5.5 Valores atípicos

In [ ]:
q1 = df["rpm"].quantile(0.25)
q3 = df["rpm"].quantile(0.75)
iqr = q3 - q1
atipicos = (df["rpm"] < q1 - 1.5 * iqr) | (df["rpm"] > q3 + 1.5 * iqr)

print(f"Límites del IQR: {q1 - 1.5 * iqr:.2f} a {q3 + 1.5 * iqr:.2f} USD por milla")
print(f"Registros atípicos: {atipicos.sum():,} ({atipicos.mean():.2%})")
print(f"Distancia promedio de los atípicos: {df.loc[atipicos, 'distance'].mean():,.0f} millas contra {df['distance'].mean():,.0f} del total")

In [ ]:
df.nlargest(5, "rpm")[["date", "lane", "distance", "midpoint", "rpm"]]

Los atípicos de la tarifa por milla se concentran en lanes cortas, donde el cargo mínimo hace que el costo por milla sea muy alto, como es el caso de South New Jersey a Philadelphia. Por otro lado, las tarifas por carga mayores a 10,000 USD se concentran en 2021, 2022 y 2026, que fueron episodios reales de precios altos en el mercado. Por lo tanto, decidimos no eliminar los atípicos, ya que quitarlos haría que el modelo aprendiera un mercado más tranquilo del que realmente existe; en su lugar, usamos la mediana al agregar por mes para que una semana extrema no mueva demasiado el valor mensual.

### 5.6 Categorías inconsistentes

In [ ]:
# Ejemplo: un mismo producto escrito de varias formas
manzanas_peras = df["commodity"].str.contains("APPLES") & df["commodity"].str.contains("PEARS")
df.loc[manzanas_peras, "commodity"].value_counts().head(6)

In [ ]:
print("Registros con el error de dedo BLUBERRIES:", (df["commodity"] == "BLUBERRIES").sum())
print("Productos distintos:", df["commodity"].nunique())

In [ ]:
# Ejemplo: un mismo origen con distintos nombres a lo largo del tiempo
df[df["origin"].str.contains("IMPERIAL")].groupby("origin")["date"].agg(["min", "max", "count"]).sort_values("min")

Encontramos dos tipos de inconsistencia. En "commodity", el mismo producto aparece escrito de varias formas, como "APPLES AND PEARS", "APPLES/PEARS", "APPLES, PEARS" o "APPLES AND PEARS." con punto al final, y también hay errores de dedo como "BLUBERRIES". Sin embargo, decidimos no corregirlos porque vamos a agrupar todos los productos de cada lane en un solo valor mensual, por lo que "commodity" no entra al modelo y corregirlo no cambiaría el resultado.

En "origin", USDA cambia el nombre de algunas zonas con el tiempo, como pasa con el Valle Imperial, que tiene varios nombres según la época. Esto puede generar que una misma lane real aparezca partida en dos series más cortas. No las unimos de forma automática porque en algunos casos el cambio de nombre también cambia el área que cubre el origen, así que lo manejamos eligiendo para el modelo solo lanes que estén activas recientemente (sección 6.4).

### 5.7 Frecuencia de los datos

In [ ]:
fechas = pd.Series(df["date"].unique()).sort_values()
print("Días entre una fecha y la siguiente:")
print(fechas.diff().dt.days.value_counts())

El calendario es semanal, ya que casi todas las fechas están separadas por 7 días, y los saltos más grandes corresponden a semanas en las que no hubo publicación. Como el diésel de la EIA tiene su corte en otro día de la semana, las dos fuentes se unen por mes y no por fecha exacta.

### 5.8 Resumen del diagnóstico

| Problema | ¿Se encontró? | Cómo lo verificamos | Posible efecto en el modelo | Qué hicimos |
|---|---|---|---|---|
| Faltantes | Sí, menos del 1% | isna() por columna | Filas sin target | Quitar filas sin "midpoint" o sin "distance"; imputar "availability" durante el modelado |
| Duplicados | Sí, por llave | duplicated() exacto y por llave | Doble peso de una lane | Eliminar duplicados por llave |
| Tipos incorrectos | Sí, todo llega como texto | info() y nulos generados al convertir | No se pueden hacer cálculos | Convertir a fecha y número |
| Registros inválidos | 6 "midpoint" fuera de su rango; 45% no es el promedio | Reglas de negocio | Target imposible o mal entendido | Eliminar los 6 y documentar la definición |
| Atípicos | Sí, lanes cortas y años de choque | Regla del IQR | Pueden dominar el error | Conservarlos y usar la mediana mensual |
| Categorías inconsistentes | Sí, en productos y orígenes | Comparación de textos | Series partidas | No corregir productos; elegir lanes activas |
| Frecuencia | Semanal con huecos | Diferencia entre fechas | Rezagos que se saltan meses | Rezagos por fecha, no por posición (sección 6.3) |

## 6. Limpieza y preparación

En esta sección solo aplicamos transformaciones que no aprenden nada de los datos, como quitar duplicados o convertir formatos, o que usan únicamente información del pasado, como los rezagos. Las transformaciones que sí aprenden parámetros, como calcular una mediana para imputar o ajustar un escalador, las vamos a ajustar solo con el conjunto de entrenamiento en el siguiente entregable, para no filtrar información del futuro. El dataset original ("df_raw" y el archivo de "data/raw") no se modifica.

### 6.1 Limpieza de registros

In [ ]:
datos = df.copy()
print(f"Registros iniciales: {len(datos):,}")

datos = datos.drop_duplicates(subset=llave)
print(f"Después de quitar duplicados por llave: {len(datos):,}")

datos = datos[~((datos["midpoint"] < datos["weeklow"]) | (datos["midpoint"] > datos["weekhigh"]))]
print(f"Después de quitar midpoint fuera de rango: {len(datos):,}")

datos = datos.dropna(subset=["midpoint", "distance"])
print(f"Después de quitar filas sin midpoint o sin distance: {len(datos):,}")

print(f"Se eliminó en total el {1 - len(datos) / len(df):.2%} de los registros")

Quitamos los duplicados porque daban doble peso a una lane, los 6 registros fuera de rango porque son imposibles y las filas sin "midpoint" o sin "distance" porque no tienen target o no se pueden comparar con las demás. No imputamos el target porque eso significaría inventar las respuestas que el modelo tiene que aprender. En total se elimina un porcentaje pequeño de los registros, por lo que la muestra no cambia de forma importante.

### 6.2 Agregación mensual por lane

Como nuestra pregunta es mensual, resumimos cada lane por mes usando la mediana de las tarifas semanales de todos sus productos.

In [ ]:
datos["mes"] = datos["date"].dt.to_period("M")

mensual = datos.groupby(["origin", "destination", "lane", "mes"]).agg(
    tarifa=("midpoint", "median"),
    rpm=("rpm", "median"),
    disponibilidad=("availability", "mean"),
    distancia=("distance", "median"),
).reset_index()

print(f"Lane-meses: {len(mensual):,} | lanes: {mensual['lane'].nunique():,}")
mensual.head()

### 6.2.1 Solo meses completos

Como USDA publica una vez por semana, el último mes descargado puede estar a medias; por ejemplo, si descargamos el 3 de noviembre, noviembre tendría solo una semana. Si dejáramos ese mes, su mediana se calcularía con menos semanas que los demás y lo estaríamos evaluando como si fuera un mes completo. La regla que usamos es sencilla: el último mes se considera completo solo si la siguiente semana que publicaría USDA (la última fecha más 7 días) ya cae en el mes siguiente; si todavía cae en el mismo mes, ese mes se elimina.

In [ ]:
ultima_fecha = datos["date"].max()
siguiente_publicacion = ultima_fecha + pd.Timedelta(days=7)

ultimo_mes_completo = ultima_fecha.to_period("M")
if siguiente_publicacion.to_period("M") == ultimo_mes_completo:
    ultimo_mes_completo = ultimo_mes_completo - 1

print("Última semana publicada:", ultima_fecha.date())
print("Siguiente publicación esperada:", siguiente_publicacion.date())
print("Último mes completo:", ultimo_mes_completo)

mensual = mensual[mensual["mes"] <= ultimo_mes_completo].reset_index(drop=True)
print(f"Lane-meses después de quitar el mes incompleto: {len(mensual):,}")

### 6.3 Variables con rezago

Para crear el rezago de N meses movemos la tabla N meses hacia adelante y la volvemos a unir por lane y mes. Así, cada mes recibe el valor que tuvo su lane N meses antes, y si ese mes no existe (por ejemplo, porque el origen solo reporta en temporada), el rezago queda vacío en lugar de tomar el valor de otro mes. Lo hicimos de esta forma y no con shift() porque shift() se mueve por renglones y no por meses, y como hay meses faltantes podría tomar un valor equivocado.

In [ ]:
def agregar_rezago(tabla, columna, meses):
    anterior = tabla[["lane", "mes", columna]].copy()
    anterior["mes"] = anterior["mes"] + meses
    anterior = anterior.rename(columns={columna: f"{columna}_lag{meses}"})
    return tabla.merge(anterior, on=["lane", "mes"], how="left")


mensual = agregar_rezago(mensual, "tarifa", 1)
mensual = agregar_rezago(mensual, "tarifa", 12)
mensual = agregar_rezago(mensual, "disponibilidad", 1)

In [ ]:
# El diésel de cada mes se usa para predecir el mes siguiente
diesel_mes = diesel.groupby(diesel["fecha"].dt.to_period("M"))["diesel"].mean().reset_index()
diesel_mes.columns = ["mes", "diesel_lag1"]
diesel_mes["mes"] = diesel_mes["mes"] + 1
mensual = mensual.merge(diesel_mes, on="mes", how="left")

# Variables de calendario; el seno y coseno hacen que diciembre y enero queden "cerca"
mensual["anio"] = mensual["mes"].dt.year
mensual["num_mes"] = mensual["mes"].dt.month
mensual["mes_sin"] = np.sin(2 * np.pi * mensual["num_mes"] / 12)
mensual["mes_cos"] = np.cos(2 * np.pi * mensual["num_mes"] / 12)

mensual = mensual.sort_values(["lane", "mes"]).reset_index(drop=True)

In [ ]:
# Revisión: en una lane, tarifa_lag1 debe ser la tarifa del renglón de arriba cuando los meses son seguidos
lane_ejemplo = mensual["lane"].value_counts().index[0]
mensual.loc[mensual["lane"] == lane_ejemplo, ["mes", "tarifa", "tarifa_lag1", "tarifa_lag12", "disponibilidad_lag1", "diesel_lag1"]].tail(6)

### 6.4 Selección de lanes para el modelo

Elegimos las lanes que tienen al menos un dato en los últimos 12 meses, para que el pronóstico sirva hoy, y que tienen al menos 36 meses con dato, para que existan los rezagos de 12 meses y haya suficiente historia para aprender.

In [ ]:
ultimo_mes = mensual["mes"].max()
resumen_lanes = mensual.groupby("lane").agg(meses_con_dato=("tarifa", "count"), ultimo_mes=("mes", "max"))

lanes_modelo = resumen_lanes[(resumen_lanes["ultimo_mes"] > ultimo_mes - 12) &
                             (resumen_lanes["meses_con_dato"] >= 36)].index
modelo = mensual[mensual["lane"].isin(lanes_modelo)].copy()

print("Último mes con datos:", ultimo_mes)
print(f"Lanes totales: {mensual['lane'].nunique()} | lanes seleccionadas: {len(lanes_modelo)}")
print(f"Lane-meses para el modelo: {len(modelo):,}")

### 6.5 División temporal

Separamos los datos por fecha y no al azar, porque en la vida real siempre vamos a pronosticar el futuro con información del pasado. Proponemos entrenar hasta 2023, validar con 2024 y dejar como prueba de enero de 2025 hasta el último mes completo (sección 6.2.1), de modo que el modelo siempre se evalúe con meses cerrados.

In [ ]:
modelo["conjunto"] = "1. entrenamiento"
modelo.loc[modelo["mes"] >= pd.Period("2024-01", freq="M"), "conjunto"] = "2. validacion"
modelo.loc[modelo["mes"] >= pd.Period("2025-01", freq="M"), "conjunto"] = "3. prueba"

modelo.groupby("conjunto").agg(filas=("tarifa", "size"), lanes=("lane", "nunique"),
                               desde=("mes", "min"), hasta=("mes", "max"))

In [ ]:
# Ejemplo de cómo se ajustará la imputación: la mediana se calcula SOLO con entrenamiento
entrenamiento = modelo[modelo["conjunto"] == "1. entrenamiento"]
mediana_entrenamiento = entrenamiento["disponibilidad_lag1"].median()
print("Mediana de disponibilidad_lag1 en entrenamiento:", mediana_entrenamiento)
print("Faltantes por conjunto:")
print(modelo.groupby("conjunto")["disponibilidad_lag1"].apply(lambda s: s.isna().sum()))

Este valor es el que se usará para llenar los faltantes de "disponibilidad_lag1" en los tres conjuntos durante el modelado. Si lo calculáramos con todos los datos, el modelo estaría usando información de 2025 y 2026 al entrenar con años anteriores. Lo mismo aplica para el escalamiento de las variables numéricas y la codificación one-hot de "origin" y "destination", que se ajustarán dentro del pipeline de entrenamiento en el siguiente entregable.

In [ ]:
mensual.assign(mes=mensual["mes"].astype(str)).to_csv(CARPETA_PROC / "lanes_mensual.csv", index=False)
modelo.assign(mes=modelo["mes"].astype(str)).to_csv(CARPETA_PROC / "lanes_modelo.csv", index=False)
print("Archivos guardados en", CARPETA_PROC)

## 7. Variables para el modelado y fuga de información

### 7.1 Entradas propuestas

| Variable | Por qué se relaciona con la pregunta | ¿Se conoce al momento de pronosticar? |
|---|---|---|
| "tarifa_lag1" | El mejor indicio del mes siguiente es el nivel actual, ya que el mercado tiene inercia | Sí, el mes ya cerró |
| "tarifa_lag12" | Captura la estacionalidad con el mismo mes del año anterior | Sí |
| "distancia" | Determina el nivel de la tarifa por carga (sección 4.5) | Sí, es fija |
| "origin" y "destination" | Cada lane tiene su propio nivel y comportamiento | Sí |
| "disponibilidad_lag1" | La escasez de camiones del mes anterior puede anticipar alzas | Sí, el mes ya cerró |
| "diesel_lag1" | El combustible es uno de los costos principales del transporte (sección 4.8) | Sí |
| "anio", "mes_sin" y "mes_cos" | Tendencia y estacionalidad (secciones 4.3 y 4.4) | Sí, es el calendario |

### 7.2 Variables que provocarían fuga de información

Las variables "weeklow" y "weekhigh" del mismo mes no se pueden usar porque se publican al mismo tiempo que la tarifa que queremos predecir, mientras que "rpm" es directamente la tarifa dividida entre la distancia. De la misma forma, la disponibilidad y el diésel del mismo mes todavía no se conocen cuando se hace el pronóstico, por lo que solo los usamos con un mes de rezago. La siguiente celda demuestra el problema: si usáramos "rpm", el modelo podría reconstruir el target casi sin error, algo que sería imposible en la realidad.

In [ ]:
train = entrenamiento.dropna(subset=["tarifa_lag1"])

error_con_fuga = (train["rpm"] * train["distancia"] - train["tarifa"]).abs().mean()
error_baseline = (train["tarifa_lag1"] - train["tarifa"]).abs().mean()
print(f"Error promedio usando rpm del mismo mes (fuga): {error_con_fuga:,.0f} USD")
print(f"Error promedio repitiendo la tarifa del mes anterior: {error_baseline:,.0f} USD")

In [ ]:
# Correlación de las entradas con el target, calculada solo con entrenamiento
entradas = ["tarifa_lag1", "tarifa_lag12", "distancia", "disponibilidad_lag1", "diesel_lag1", "anio", "mes_sin", "mes_cos"]
correlaciones = train[entradas + ["tarifa"]].corr()["tarifa"].drop("tarifa").sort_values()

correlaciones.plot.barh(color="steelblue")
plt.title("Correlación de cada entrada con la tarifa (solo entrenamiento)")
plt.show()

Un error casi de cero usando "rpm" no significaría que el modelo es bueno, sino que estaría haciendo trampa. En cambio, repetir la tarifa del mes anterior nos da un error honesto que funciona como baseline, y el modelo del siguiente entregable tendrá que superarlo para que valga la pena. La gráfica de correlaciones, calculada solo con entrenamiento para no filtrar información de los otros conjuntos, confirma que los rezagos de la tarifa y la distancia son las entradas más relacionadas con el target, como esperábamos por el EDA.

## 8. Conclusiones

### 8.1 Hallazgos principales

El primer hallazgo es que el mercado tiene una tendencia creciente con cambios de nivel muy marcados, ya que la tarifa por milla pasó de 1.63 USD en 2000 a 4.27 en 2022, bajó a alrededor de 3.4 entre 2023 y 2025 y vuelve a subir en 2026 hasta cerca de 4.0. Esto hace que un modelo entrenado solo con años estables podría subestimar los choques, por lo que es necesario validar en orden temporal y apoyarse en los rezagos.

El segundo hallazgo es que existe una estacionalidad moderada pero constante, donde julio es cerca de 5.5% más caro que el promedio de su año y febrero y marzo alrededor de 6% más baratos, lo cual justifica incluir el mes y el rezago de 12 meses.

El tercer hallazgo es que la distancia explica la mayor parte del nivel de la tarifa por carga, mientras que la tarifa por milla baja conforme la lane es más larga (4.73 USD en lanes de menos de 700 millas contra 2.33 en lanes de 2,000 millas o más), por lo que el modelo tiene que trabajar lane por lane.

Adicionalmente, encontramos que la cobertura cambió mucho en el tiempo, ya que los registros por año bajaron de cerca de 10,000 a 3,875 y varios orígenes se dejaron de reportar o cambiaron de nombre, y que la calidad general es buena, con faltantes menores al 1%, duplicados causados por la doble asignación de región y una diferencia entre la documentación y los datos en la definición de "midpoint".

### 8.2 ¿Los datos permiten abordar el problema?

Sí, aunque solo para las lanes que siguen activas y que tienen suficiente historia. Tenemos 26 años de datos semanales, 112 lanes con información en 2026 y, después de agregar por mes y filtrar, varios miles de lane-meses para entrenar, validar y probar (sección 6.4). Por otro lado, los datos no alcanzan para pronosticar lanes nuevas, orígenes que ya no se reportan o meses fuera de temporada de los orígenes estacionales.

### 8.3 Limitaciones

Las tarifas son promedios de mercado publicados por USDA y no las tarifas que realmente paga la empresa, por lo que la comparación entre spot y contrato necesitaría datos internos. Asimismo, el diésel es un promedio nacional, "availability" es un indicador ordinal reportado por origen y no por lane, algunos orígenes renombrados parten series que en realidad son la misma, y 2026 todavía es un año incompleto.

### 8.4 Cómo se medirá el beneficio

Es importante separar dos cosas. En este proyecto, el beneficio que podemos medir es un **beneficio estimado con datos de mercado**: con el conjunto de prueba armamos un presupuesto simulado, donde el costo de cada lane-mes es la tarifa que pronostica el modelo, y lo comparamos contra la tarifa que realmente publicó USDA para ese mes. Esto nos permite calcular el MAE en USD por carga y el error porcentual del presupuesto total, siempre con meses completos.

Por otro lado, la desviación contra el **gasto real** de la empresa solo se podrá comprobar en una etapa posterior, cuando se tengan sus facturas de flete o sus tarifas contratadas, ya que esos datos son internos y no forman parte de este proyecto. Por lo tanto, los resultados que presentemos deben entenderse como una estimación del beneficio, y la validación con gasto real queda como el siguiente paso para la empresa.

### 8.5 Cambios necesarios para el modelado

Durante el modelado vamos a ajustar la imputación, el escalamiento y la codificación únicamente con el conjunto de entrenamiento, usando una división temporal en lugar de una aleatoria. También vamos a comparar el modelo contra dos baselines sencillos (la tarifa del mes anterior y la del mismo mes del año anterior) usando el MAE en USD por carga, y a probar modelos que manejen bien los rezagos vacíos, como HistGradientBoostingRegressor. Por último, podría convenir modelar el cambio porcentual contra el mes anterior en lugar del nivel, para reducir el problema de que los árboles no extrapolan a tarifas más altas que las del entrenamiento.

## 9. Referencias

- U.S. Department of Agriculture, Agricultural Marketing Service. (2026). *Refrigerated Truck Rates and Availability* [Conjunto de datos]. AgTransport. https://agtransport.usda.gov/d/acar-e3r8
- U.S. Energy Information Administration. (2026). *Weekly U.S. No 2 Diesel Retail Prices* [Conjunto de datos]. https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=EMD_EPD2D_PTE_NUS_DPG&f=W
- Tyler Technologies. (s.f.). *Socrata Open Data API documentation*. https://dev.socrata.com/